In [15]:
import os
import cv2
import numpy as np
import wandb
from tqdm import tqdm

# ==========================================================
# 1. SETUP
# ==========================================================
run = wandb.init(project="Depth-Comparison", name="final_verified_scientific_run")

# Added columns for 'Mean Predicted Depth' to verify values
columns = [
    "ID", "Image", "GT_Mean_Meters", "DA_Mean_Meters", "MiDaS_Mean_Meters",
    "DA_RMSE", "MiDaS_RMSE", "DA_AbsRel", "MiDaS_AbsRel", "DA_D1", "MiDaS_D1", "Winner"
]
comparison_table = wandb.Table(columns=columns)

def compute_all_metrics(gt, pred):
    pred = np.maximum(pred, 1e-6)
    rmse = np.sqrt(np.mean((gt - pred) ** 2))
    abs_rel = np.mean(np.abs(gt - pred) / gt)
    thresh = np.maximum((gt / pred), (pred / gt))
    d1 = (thresh < 1.25).mean()
    return {"rmse": rmse, "abs_rel": abs_rel, "d1": d1}

def align_and_eval(gt_raw, pred_raw):
    # Resize
    pred_res = cv2.resize(pred_raw.astype(np.float32), (gt_raw.shape[1], gt_raw.shape[0]))
    mask = (gt_raw > 1.0) & (gt_raw < 50.0)
    if not np.any(mask): return None
    
    # LINEARIZATION (Crucial Step)
    p_norm = (pred_res - pred_res.min()) / (pred_res.max() - pred_res.min() + 1e-8)
    pred_inv = 1.0 / (p_norm + 0.05) 
    
    # MEDIAN SCALING
    scale = np.median(gt_raw[mask]) / np.median(pred_inv[mask])
    aligned = pred_inv * scale
    
    metrics = compute_all_metrics(gt_raw[mask], aligned[mask])
    return metrics, aligned

# ==========================================================
# 2. EVALUATION LOOP
# ==========================================================
image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png'))]
da_wins = 0
gt_all, da_all, mi_all = [], [], [] # For global scatter plots

for img_name in tqdm(image_files):
    unique_id = img_name.split('_image')[0]
    
    # [Load your images here: gt, da_r, mi_r]
    # gt = cv2.imread(lp, -1) etc.

    da_eval = align_and_eval(gt, da_r)
    mi_eval = align_and_eval(gt, mi_r)

    if da_eval and mi_eval:
        d_m, d_a = da_eval
        m_m, m_a = mi_eval
        
        # Calculate Mean Values for the Table
        mask = (gt > 1.0) & (gt < 50.0)
        gt_mean = np.mean(gt[mask])
        da_mean = np.mean(d_a[mask])
        mi_mean = np.mean(m_a[mask])

        winner = "DA-V2" if d_m['abs_rel'] < m_m['abs_rel'] else "MiDaS"
        if winner == "DA-V2": da_wins += 1

        comparison_table.add_data(
            unique_id, wandb.Image(os.path.join(IMG_DIR, img_name)),
            round(gt_mean, 2), round(da_mean, 2), round(mi_mean, 2),
            round(d_m['rmse'], 3), round(m_m['rmse'], 3),
            round(d_m['abs_rel'], 3), round(m_m['abs_rel'], 3),
            round(d_m['d1'], 3), round(m_m['d1'], 3),
            winner
        )
        
        # Log values for global charts
        wandb.log({
            "val/rmse_da": d_m['rmse'], "val/rmse_mi": m_m['rmse'],
            "val/absrel_da": d_m['abs_rel'], "val/absrel_mi": m_m['abs_rel'],
            "val/d1_da": d_m['d1'], "val/d1_mi": m_m['d1']
        })

# ==========================================================
# 3. 10+ CHARTS LOGGING
# ==========================================================
wandb.log({"Results/Detailed_Metrics_Table": comparison_table})

data_np = np.array(comparison_table.data)
# Convert to float for safety
da_rmse = data_np[:, 5].astype(float)
mi_rmse = data_np[:, 6].astype(float)
da_rel = data_np[:, 7].astype(float)
mi_rel = data_np[:, 8].astype(float)

# CHART 1: Average RMSE
wandb.log({"Charts/Avg_RMSE": wandb.plot.bar(wandb.Table(data=[["DA-V2", np.mean(da_rmse)], ["MiDaS", np.mean(mi_rmse)]], columns=["Model", "RMSE"]), "Model", "RMSE", title="Mean RMSE")})

# CHART 2: Average AbsRel
wandb.log({"Charts/Avg_AbsRel": wandb.plot.bar(wandb.Table(data=[["DA-V2", np.mean(da_rel)], ["MiDaS", np.mean(mi_rel)]], columns=["Model", "AbsRel"]), "Model", "AbsRel", title="Mean Abs_Rel")})

# CHART 3: D1 Accuracy
wandb.log({"Charts/Avg_D1": wandb.plot.bar(wandb.Table(data=[["DA-V2", np.mean(data_np[:,9].astype(float))], ["MiDaS", np.mean(data_np[:,10].astype(float))]], columns=["Model", "D1"]), "Model", "D1", title="Mean Delta 1")})

# CHART 4-9: Histograms (Automatically generated by the 'val/' logs in the loop)

# CHART 10: Win Rate Bar
wandb.log({"Charts/Total_Wins": wandb.plot.bar(wandb.Table(data=[["DA-V2", da_wins], ["MiDaS", len(image_files)-da_wins]], columns=["Model", "Wins"]), "Model", "Wins", title="Win Count")})

run.finish()

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


100%|████████████████████████████████████████████████████████████████████████████████| 101/101 [00:03<00:00, 30.00it/s]


val/absrel_da,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/absrel_mi,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/d1_da,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/d1_mi,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/rmse_da,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/rmse_mi,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/absrel_da,0.82045
val/absrel_mi,2.06954
val/d1_da,0.08518
val/d1_mi,0.06302
val/rmse_da,1.51271


In [5]:
# Pick one image and one pixel (center of the image)
h, w = gt.shape
cy, cx = h // 2, w // 2

print(f"Diagnostic for Center Pixel ({cx}, {cy}):")
print(f"LiDAR Value (Meters): {gt[cy, cx]:.2f}")
print(f"DA-V2 Raw Value:      {da[cy, cx]}")
print(f"MiDaS Raw Value:     {midas[cy, cx]}")

# CHECK: When you move closer to an object, does the LiDAR value decrease 
# and the Model value increase? If so, the model is INVERSE depth.

Diagnostic for Center Pixel (96, 128):
LiDAR Value (Meters): 0.93
DA-V2 Raw Value:      3
MiDaS Raw Value:     39
